In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("asset_id", "")
dbutils.widgets.text("assignment_id", "")
dbutils.widgets.text("rule_code", "")
dbutils.widgets.text("version_number", "")
dbutils.widgets.text("target_column_id", "")
dbutils.widgets.text("asset_execution_id", "")
dbutils.widgets.text("parameter_values", "")
dbutils.widgets.text("sql_expression", "")
dbutils.widgets.text("threshold_value", "")
dbutils.widgets.text("evaluation_operation", "")


asset_id = dbutils.widgets.get("asset_id").strip()
assignment_id = dbutils.widgets.get("assignment_id").strip()
rule_code = dbutils.widgets.get("rule_code").strip()
version_number = dbutils.widgets.get("version_number").strip()
target_column_id = dbutils.widgets.get("target_column_id").strip()
asset_execution_id = dbutils.widgets.get(
    "asset_execution_id"
).strip()
parameter_values = dbutils.widgets.get("parameter_values").strip()
sql_expression = dbutils.widgets.get("sql_expression").strip()
threshold_value = dbutils.widgets.get("threshold_value").strip()
evaluation_operation = dbutils.widgets.get("evaluation_operation").strip()


print("====================================")
print("RULE EXECUTION INPUT")
print("====================================")

print(f"asset_id             : {asset_id}")
print(f"assignment_id        : {assignment_id}")
print(f"rule_code            : {rule_code}")
print(f"version_number       : {version_number}")
print(f"target_column_id     : {target_column_id}")
print(f"parameter_values     : {parameter_values}")
print(f"sql_expression       : {sql_expression}")
print(f"threshold_value      : {threshold_value}")
print(f"evaluation_operation : {evaluation_operation}")

RULE EXECUTION INPUT
asset_id             : 1
assignment_id        : 926
rule_code            : V08
version_number       : 1
target_column_id     : 504
parameter_values     : {"format_definition":"^AST[0-9]{8}$"}
sql_expression       : 
threshold_value      : 99.000000
evaluation_operation : >=


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import json
from decimal import Decimal
import pandas as pd
import re


CATALOG = "mastercard_card_241614"
SCHEMA = "default"

RULES_TABLE = f"{CATALOG}.{SCHEMA}.rules_m"
ASSIGNMENTS_TABLE = f"{CATALOG}.{SCHEMA}.rules_assignments_m"
ASSETS_TABLE = f"{CATALOG}.{SCHEMA}.assets_m"
ASSET_COLUMNS_TABLE = f"{CATALOG}.{SCHEMA}.asset_columns_m"

SOURCE_EXCEL = "/Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx"

SOURCE_VIEW = "dq_source_data"


print("====================================")
print("RULE EXECUTION CONFIGURATION")
print("====================================")

print(f"Rules table        : {RULES_TABLE}")
print(f"Assignments table  : {ASSIGNMENTS_TABLE}")
print(f"Assets table       : {ASSETS_TABLE}")
print(f"Columns table      : {ASSET_COLUMNS_TABLE}")
print(f"Source Excel       : {SOURCE_EXCEL}")
print(f"Source temp view   : {SOURCE_VIEW}")

RULE EXECUTION CONFIGURATION
Rules table        : mastercard_card_241614.default.rules_m
Assignments table  : mastercard_card_241614.default.rules_assignments_m
Assets table       : mastercard_card_241614.default.assets_m
Columns table      : mastercard_card_241614.default.asset_columns_m
Source Excel       : /Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx
Source temp view   : dq_source_data


In [0]:
print("====================================")
print("VALIDATING RULE EXECUTION INPUT")
print("====================================")

required_inputs = {
    "asset_id": asset_id,
    "assignment_id": assignment_id,
    "rule_code": rule_code,
    "version_number": version_number
}

missing_inputs = [
    name
    for name, value in required_inputs.items()
    if value == ""
]

if missing_inputs:
    raise ValueError(
        f"Missing required rule execution inputs: {missing_inputs}"
    )


try:
    asset_id_int = int(asset_id)
    assignment_id_int = int(assignment_id)
    version_number_int = int(version_number)
except ValueError:
    raise ValueError(
        "asset_id, assignment_id and version_number must be valid integers."
    )


if target_column_id != "":
    try:
        target_column_id_int = int(target_column_id)
    except ValueError:
        raise ValueError(
            "target_column_id must be a valid integer when provided."
        )
else:
    target_column_id_int = None


print("Input validation PASSED")
print(f"asset_id         : {asset_id_int}")
print(f"assignment_id    : {assignment_id_int}")
print(f"rule_code        : {rule_code}")
print(f"version_number   : {version_number_int}")
print(f"target_column_id : {target_column_id_int}")

VALIDATING RULE EXECUTION INPUT
Input validation PASSED
asset_id         : 1
assignment_id    : 926
rule_code        : V08
version_number   : 1
target_column_id : 504


In [0]:
print("====================================")
print("RESOLVING RULE ASSIGNMENT")
print("====================================")

rules_assignments_df = spark.table(ASSIGNMENTS_TABLE)

assignment_df = (
    rules_assignments_df
    .filter(F.col("assignment_id") == assignment_id_int)
    .filter(F.col("asset_id") == asset_id_int)
    .filter(F.col("rule_code") == rule_code)
    .filter(F.col("version_number") == version_number_int)
    .filter(F.col("is_enabled") == True)
    .filter(F.col("is_current") == True)
)

assignment_count = assignment_df.count()

if assignment_count == 0:
    raise ValueError(
        "No enabled/current rule assignment found.\n"
        f"asset_id={asset_id_int}, "
        f"assignment_id={assignment_id_int}, "
        f"rule_code={rule_code}, "
        f"version_number={version_number_int}. "
        "The assignment may be disabled, historical, or invalid."
    )

if assignment_count > 1:
    raise ValueError(
        "Multiple enabled/current rule assignments found. "
        f"assignment_id={assignment_id_int} should uniquely identify "
        "one current assignment."
    )

assignment = assignment_df.first()

print("Rule assignment resolved successfully")
print("------------------------------------")
print(f"assignment_id        : {assignment['assignment_id']}")
print(f"asset_id             : {assignment['asset_id']}")
print(f"rule_code            : {assignment['rule_code']}")
print(f"version_number       : {assignment['version_number']}")
print(f"target_column_id     : {assignment['target_column_id']}")
print(f"target_type          : {assignment['target_type']}")
print(f"parameter_values     : {assignment['parameter_values']}")
print(f"sql_expression       : {assignment['sql_expression']}")
print(f"threshold_value      : {assignment['threshold_value']}")
print(f"evaluation_operation : {assignment['evaluation_operation']}")
print(f"is_enabled           : {assignment['is_enabled']}")
print(f"is_current           : {assignment['is_current']}")


RESOLVING RULE ASSIGNMENT
Rule assignment resolved successfully
------------------------------------
assignment_id        : 926
asset_id             : 1
rule_code            : V08
version_number       : 1
target_column_id     : 504
target_type          : COLUMN
parameter_values     : {"format_definition":"^AST[0-9]{8}$"}
sql_expression       : None
threshold_value      : 99.000000
evaluation_operation : >=
is_enabled           : True
is_current           : True


In [0]:
print("====================================")
print("RESOLVING RULE METADATA")
print("====================================")


rules_df = spark.table(RULES_TABLE)


rule_metadata_df = (
    rules_df
    .filter(F.col("rule_code") == rule_code)
    .filter(F.col("is_active") == True)
)


rule_metadata_count = rule_metadata_df.count()


if rule_metadata_count == 0:
    raise ValueError(
        f"No active rule metadata found for rule_code='{rule_code}'."
    )


if rule_metadata_count > 1:
    raise ValueError(
        f"Multiple active rule definitions found for "
        f"rule_code='{rule_code}'."
    )


rule_metadata = rule_metadata_df.first()


dimension = rule_metadata["dimension"]
rule_name = rule_metadata["rule_name"]
rule_type = rule_metadata["rule_type"]
description = rule_metadata["description"]
target_type = rule_metadata["target_type"]
parameter_definition = rule_metadata["parameter_definition"]
sql_query_template = rule_metadata["sql_query_template"]
default_threshold = rule_metadata["default_threshold"]
data_type = rule_metadata["data_type"]
semantic_type = rule_metadata["semantic_type"]


print("Rule metadata resolved successfully")
print("------------------------------------")

print(f"rule_code            : {rule_metadata['rule_code']}")
print(f"dimension            : {dimension}")
print(f"rule_name            : {rule_name}")
print(f"rule_type            : {rule_type}")
print(f"target_type          : {target_type}")
print(f"data_type            : {data_type}")
print(f"semantic_type        : {semantic_type}")
print(f"default_threshold    : {default_threshold}")
print(f"parameter_definition : {parameter_definition}")
print("------------------------------------")
print("SQL QUERY TEMPLATE")
print("------------------------------------")
print(sql_query_template)

RESOLVING RULE METADATA
Rule metadata resolved successfully
------------------------------------
rule_code            : V08
dimension            : VALIDITY
rule_name            : Format compliance %
rule_type            : PARAMETERIZED
target_type          : COLUMN
data_type            : STRING
semantic_type        : CODE/ID/EMAIL/PHONE
default_threshold    : 99.000000
parameter_definition : {
  "parameters": [
    {
      "name": "format_definition",
      "label": "Format definition",
      "control": "regex",
      "required": true
    }
  ]
}
------------------------------------
SQL QUERY TEMPLATE
------------------------------------

SELECT
    100.0 * COUNT_IF(
        REGEXP_LIKE({column}, {format_definition})
    )
    / NULLIF(COUNT({column}), 0) AS metric_value,

    COUNT_IF(
        REGEXP_LIKE({column}, {format_definition})
    ) AS passed_count,

    COUNT_IF(
        NOT REGEXP_LIKE({column}, {format_definition})
    ) AS failed_count,

    COUNT({column}) AS total_count

In [0]:
print("====================================")
print("RESOLVING ASSET AND TARGET COLUMN")
print("====================================")

assets_df = spark.table(ASSETS_TABLE)

asset_df = (
    assets_df
    .filter(F.col("asset_id") == asset_id_int)
    .filter(F.col("is_active") == True)
)

asset_count = asset_df.count()

if asset_count == 0:
    raise ValueError(
        f"No active asset found for asset_id={asset_id_int}."
    )

if asset_count > 1:
    raise ValueError(
        f"Multiple active assets found for asset_id={asset_id_int}."
    )

asset = asset_df.first()

asset_name = asset["asset_name"]
asset_type = asset["asset_type"]
asset_s3_path = asset["s3_path"]
asset_domain_id = asset["domain_id"]
asset_sub_domain_id = asset["sub_domain_id"]

print("Asset resolved")
print("------------------------------------")
print(f"asset_id       : {asset_id_int}")
print(f"asset_name     : {asset_name}")
print(f"asset_type     : {asset_type}")
print(f"s3_path        : {asset_s3_path}")
print(f"domain_id      : {asset_domain_id}")
print(f"sub_domain_id  : {asset_sub_domain_id}")

target_column_name = None
target_column_data_type = None
target_column_nullable = None
target_column_metadata = None

if target_column_id_int is not None:

    asset_columns_df = spark.table(ASSET_COLUMNS_TABLE)

    target_column_df = (
        asset_columns_df
        .filter(F.col("column_id") == target_column_id_int)
        .filter(F.col("asset_id") == asset_id_int)
    )

    target_column_count = target_column_df.count()

    if target_column_count == 0:
        raise ValueError(
            "Target column could not be resolved.\n"
            f"column_id={target_column_id_int}, "
            f"asset_id={asset_id_int}"
        )

    if target_column_count > 1:
        raise ValueError(
            "Multiple target columns found for the same "
            "column_id and asset_id."
        )

    target_column = target_column_df.first()

    target_column_name = target_column["column_name"]
    target_column_data_type = target_column["data_type"]
    target_column_nullable = target_column["is_nullable"]
    target_column_metadata = target_column["metadata"]

    print("")
    print("Target column resolved")
    print("------------------------------------")
    print(f"column_id       : {target_column_id_int}")
    print(f"column_name     : {target_column_name}")
    print(f"data_type       : {target_column_data_type}")
    print(f"is_nullable     : {target_column_nullable}")
    print(f"metadata        : {target_column_metadata}")

else:

    print("")
    print("No target column specified.")
    print("Rule will execute at asset/table level.")

# Validate assignment target type against the actual supplied target.
assignment_target_type = assignment["target_type"]

if assignment_target_type is not None:
    normalized_assignment_target_type = (
        str(assignment_target_type).strip().upper()
    )
else:
    normalized_assignment_target_type = None

if normalized_assignment_target_type in {
    "COLUMN",
    "COLUMN / CROSS-COLUMN"
} and target_column_id_int is None:
    raise ValueError(
        f"Assignment {assignment_id_int} has target_type="
        f"'{assignment_target_type}' but no target_column_id."
    )

if normalized_assignment_target_type == "TABLE" and target_column_id_int is not None:
    raise ValueError(
        f"Assignment {assignment_id_int} has target_type='TABLE' "
        "but a target_column_id was supplied."
    )

# Cross-asset fields are intentionally not executed yet.
if assignment["target_asset_id"] is not None:
    raise NotImplementedError(
        "Cross-asset rule execution is not enabled in this prototype. "
        f"assignment_id={assignment_id_int} contains target_asset_id."
    )

if assignment["target_asset_column_id"] is not None:
    raise NotImplementedError(
        "Cross-asset column execution is not enabled in this prototype. "
        f"assignment_id={assignment_id_int} contains target_asset_column_id."
    )


RESOLVING ASSET AND TARGET COLUMN
Asset resolved
------------------------------------
asset_id       : 1
asset_name     : 03_ISS_account_status
asset_type     : TABLE
s3_path        : source/ISSUER/ISSUER_ACCOUNT_MANAGEMENT/03_iss_account_status
domain_id      : 3
sub_domain_id  : 5

Target column resolved
------------------------------------
column_id       : 504
column_name     : STATUS_ID
data_type       : STRING
is_nullable     : False
metadata        : {"semantic_type":"ID"}


In [0]:
# ============================================================
# VALIDATE TARGET COLUMN
# ============================================================

if target_column_id_int is not None:

    target_column_rows = (
        asset_columns_df
        .filter(
            (F.col("asset_id") == asset_id_int) &
            (F.col("column_id") == target_column_id_int)
        )
        .collect()
    )

    if not target_column_rows:

        raise ValueError(
            f"Target column ID {target_column_id_int} "
            f"does not exist for asset_id {asset_id_int}. "
            f"Assignment {assignment_id_int} "
            f"must be corrected in rules_assignments_m."
        )

    target_column_name = target_column_rows[0]["column_name"]

    print(
        f"Target column resolved : "
        f"{target_column_id_int} -> {target_column_name}"
    )

else:

    target_column_name = None

    print("Rule is table-level.")

Target column resolved : 504 -> STATUS_ID


In [0]:
# ============================================================
# LOADING SOURCE DATA
# ============================================================

print("====================================")
print("LOADING SOURCE DATA")
print("====================================")


# --------------------------------------------------
# 1. Load the asset's Excel sheet
# --------------------------------------------------

print(f"Excel file : {SOURCE_EXCEL}")
print(f"Sheet name : {asset_name}")

pdf = pd.read_excel(
    SOURCE_EXCEL,
    sheet_name=asset_name
)

print(f"Rows loaded from Excel : {len(pdf)}")
print(f"Columns found           : {len(pdf.columns)}")


# --------------------------------------------------
# 2. Preserve raw source values
# --------------------------------------------------
#
# All source values are intentionally kept as strings.
#
# This is important because DQ rules must be able to
# distinguish:
#
#   - valid typed values
#   - malformed values
#   - NULL values
#
# For example:
#
#   "2023-03-09 07:40:53" -> valid timestamp
#   "TXN6022768040"        -> malformed timestamp
#   NULL                   -> missing value
#
# --------------------------------------------------

for col in pdf.columns:
    pdf[col] = pdf[col].apply(
        lambda x: str(x) if pd.notna(x) else None
    )


df = spark.createDataFrame(pdf)


# --------------------------------------------------
# 3. Basic source information
# --------------------------------------------------

print("")
print("Spark source DataFrame created")
print("------------------------------------")

print(f"Row count    : {df.count()}")
print(f"Column count : {len(df.columns)}")


# --------------------------------------------------
# 4. Register raw source view
# --------------------------------------------------

df.createOrReplaceTempView(SOURCE_VIEW)


print("")
print(f"Temporary view created : {SOURCE_VIEW}")


# --------------------------------------------------
# 5. Display source structure
# --------------------------------------------------

print("")
print("SOURCE SCHEMA")
print("------------------------------------")

df.printSchema()

LOADING SOURCE DATA
Excel file : /Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx
Sheet name : 03_ISS_account_status
Rows loaded from Excel : 100
Columns found           : 7

Spark source DataFrame created
------------------------------------
Row count    : 100
Column count : 7

Temporary view created : dq_source_data

SOURCE SCHEMA
------------------------------------
root
 |-- STATUS_ID: string (nullable = true)
 |-- ACCOUNT_ID: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- EFFECTIVE_DATE: string (nullable = true)
 |-- REASON_CODE: string (nullable = true)
 |-- CHANGED_BY: string (nullable = true)
 |-- DQ_FLAG: string (nullable = true)



In [0]:
print("====================================")
print("PARSING RULE PARAMETERS")
print("====================================")


# --------------------------------------------------
# 1. Initialize parameter values
# --------------------------------------------------

parsed_parameters = {}


if parameter_values in ("", "null", "NULL", "None"):
    parsed_parameters = {}

else:
    try:
        parsed_parameters = json.loads(parameter_values)

    except json.JSONDecodeError as e:
        raise ValueError(
            "parameter_values is not valid JSON.\n"
            f"Received: {parameter_values}\n"
            f"Error: {str(e)}"
        )


# --------------------------------------------------
# 2. Validate parameter structure
# --------------------------------------------------

if not isinstance(parsed_parameters, dict):
    raise ValueError(
        "parameter_values must contain a JSON object."
    )


print("Parameter parsing completed")
print("------------------------------------")
print(f"Parameters supplied : {len(parsed_parameters)}")


if parsed_parameters:
    for parameter_name, parameter_value in parsed_parameters.items():
        print(
            f"{parameter_name} : "
            f"{parameter_value}"
        )
else:
    print("No parameters supplied.")


# --------------------------------------------------
# 3. Read parameter definition from rules_m
# --------------------------------------------------

parameter_definition_obj = None


if parameter_definition not in (None, "", "null", "NULL"):

    try:
        parameter_definition_obj = json.loads(
            parameter_definition
        )

    except json.JSONDecodeError as e:
        raise ValueError(
            f"parameter_definition for rule_code='{rule_code}' "
            "is not valid JSON.\n"
            f"Error: {str(e)}"
        )


# --------------------------------------------------
# 4. Validate required parameters
# --------------------------------------------------

if parameter_definition_obj:

    if isinstance(parameter_definition_obj, dict):

        parameter_definitions = (
            parameter_definition_obj.get(
                "parameters",
                []
            )
        )

    elif isinstance(parameter_definition_obj, list):

        parameter_definitions = parameter_definition_obj

    else:

        parameter_definitions = []


    for parameter in parameter_definitions:

        if not isinstance(parameter, dict):
            continue

        parameter_name = parameter.get("name")
        required = parameter.get("required", False)

        if (
            required
            and parameter_name
            and parameter_name not in parsed_parameters
        ):
            raise ValueError(
                f"Required parameter '{parameter_name}' "
                f"is missing for rule_code='{rule_code}'."
            )


print("")
print("Parameter validation PASSED")

PARSING RULE PARAMETERS
Parameter parsing completed
------------------------------------
Parameters supplied : 1
format_definition : ^AST[0-9]{8}$

Parameter validation PASSED


In [0]:
# ============================================================
# VALIDATE REQUIRED PARAMETERS
# ============================================================

print("====================================")
print("VALIDATING RULE PARAMETERS")
print("====================================")

required_parameters = []

if parameter_definition:
    try:
        parameter_definition_json = json.loads(
            parameter_definition
        )

        required_parameters = [
            p["name"]
            for p in parameter_definition_json.get("parameters", [])
            if p.get("required") is True
        ]

    except Exception as e:
        raise ValueError(
            f"Invalid parameter_definition JSON for rule "
            f"{rule_code}: {e}"
        )


missing_parameters = []

for parameter_name in required_parameters:

    value = parsed_parameters.get(parameter_name)

    if value is None:
        missing_parameters.append(parameter_name)

    elif isinstance(value, str) and not value.strip():
        missing_parameters.append(parameter_name)


if missing_parameters:

    raise ValueError(
        f"Rule {rule_code} is missing required parameters: "
        f"{', '.join(missing_parameters)}"
    )


print("Required parameters : ", required_parameters)
print("Provided parameters : ", list(parsed_parameters.keys()))
print("Parameter validation PASSED")

VALIDATING RULE PARAMETERS
Required parameters :  ['format_definition']
Provided parameters :  ['format_definition']
Parameter validation PASSED


In [0]:
# ============================================================
# BUILD EXECUTION CONTEXT
# ============================================================

print("====================================")
print("BUILDING EXECUTION CONTEXT")
print("====================================")


# ============================================================
# 1. SAFE SQL IDENTIFIER HELPER
# ============================================================

def sql_identifier(name):

    if name is None:
        raise ValueError(
            "SQL identifier cannot be None."
        )

    name = str(name).strip()

    if not name:
        raise ValueError(
            "SQL identifier cannot be empty."
        )

    return (
        "`"
        + name.replace("`", "``")
        + "`"
    )


# ============================================================
# 2. SOURCE RELATION
# ============================================================

source_relation = SOURCE_VIEW


# ============================================================
# 3. TARGET COLUMN
# ============================================================

target_column_sql = None

if target_column_name is not None:

    target_column_sql = sql_identifier(
        target_column_name
    )


# ============================================================
# 4. RESOLVE EFFECTIVE THRESHOLD
#
# Precedence:
#   1. Notebook input
#   2. Assignment threshold
#   3. Rule default threshold
#
# NULL is valid.
#
# This is important for:
#   P01/P02/P03/P04/P05/P06/P11
#   and other measurement/profiling rules.
# ============================================================

effective_threshold = None

threshold_candidates = [

    (
        "notebook_input",
        threshold_value
    ),

    (
        "assignment",
        assignment["threshold_value"]
    ),

    (
        "rule_default",
        default_threshold
    )
]


for threshold_source, candidate in threshold_candidates:

    if candidate is None:
        continue

    candidate_string = str(candidate).strip()

    if candidate_string == "":
        continue

    try:

        effective_threshold = float(
            candidate_string
        )

    except (TypeError, ValueError):

        raise ValueError(
            f"Invalid threshold value "
            f"'{candidate}' from {threshold_source} "
            f"for rule_code='{rule_code}', "
            f"assignment_id={assignment_id_int}."
        )

    break


# ============================================================
# 5. RESOLVE EFFECTIVE EVALUATION OPERATION
#
# Precedence:
#   1. Notebook input
#   2. Assignment value
#   3. Rule default
#
# NULL is allowed when there is no threshold.
# ============================================================

effective_evaluation_operation = None

operation_candidates = [

    (
        "notebook_input",
        evaluation_operation
    ),

    (
        "assignment",
        assignment["evaluation_operation"]
    ),

    (
        "rule_default",
        rule_metadata["evaluation_operator"]
    )
]


for operation_source, candidate in operation_candidates:

    if candidate is None:
        continue

    candidate_string = str(candidate).strip()

    if candidate_string == "":
        continue

    effective_evaluation_operation = (
        candidate_string
    )

    break


# ============================================================
# 6. VALIDATE EVALUATION OPERATION
# ============================================================

allowed_operations = {
    ">",
    ">=",
    "<",
    "<=",
    "=",
    "==",
    "!="
}


if effective_threshold is not None:

    if effective_evaluation_operation is None:

        raise ValueError(
            f"Threshold {effective_threshold} exists "
            f"but no evaluation operation was resolved "
            f"for rule_code='{rule_code}', "
            f"assignment_id={assignment_id_int}."
        )

    if (
        effective_evaluation_operation
        not in allowed_operations
    ):

        raise ValueError(
            f"Unsupported evaluation operation "
            f"'{effective_evaluation_operation}' "
            f"for rule_code='{rule_code}'. "
            f"Allowed operations: "
            f"{sorted(allowed_operations)}"
        )


elif effective_evaluation_operation is not None:

    if (
        effective_evaluation_operation
        not in allowed_operations
    ):

        raise ValueError(
            f"Unsupported evaluation operation "
            f"'{effective_evaluation_operation}' "
            f"for rule_code='{rule_code}'. "
            f"Allowed operations: "
            f"{sorted(allowed_operations)}"
        )


# ============================================================
# 7. RESOLVE EFFECTIVE SQL EXPRESSION
# ============================================================

effective_sql_expression = None

if (
    sql_expression is not None
    and str(sql_expression).strip() != ""
):

    effective_sql_expression = (
        str(sql_expression).strip()
    )

elif assignment["sql_expression"] is not None:

    effective_sql_expression = (
        str(
            assignment["sql_expression"]
        ).strip()
    )


# ============================================================
# 8. BUILD EXECUTION CONTEXT
# ============================================================

execution_context = {

    # Standard SQL placeholders
    "table": source_relation,
    "asset": source_relation,
    "column": target_column_sql,

    # Asset information
    "asset_id": asset_id_int,
    "asset_name": asset_name,
    "asset_type": asset_type,

    # Assignment information
    "assignment_id": assignment_id_int,
    "rule_code": rule_code,
    "version_number": version_number_int,

    # Rule information
    "dimension": dimension,
    "rule_name": rule_name,
    "rule_type": rule_type,
    "target_type": target_type,

    # Target information
    "target_column_id": target_column_id_int,
    "target_column_name": target_column_name,
    "target_column_sql": target_column_sql,

    # Source
    "source_relation": source_relation,

    # Parameters
    "parameter_values": parsed_parameters,

    # SQL override
    "sql_expression": effective_sql_expression,

    # Evaluation
    "threshold_value": effective_threshold,
    "evaluation_operation": (
        effective_evaluation_operation
    ),

    # Metadata
    "data_type": data_type,
    "semantic_type": semantic_type
}


# ============================================================
# 9. SOURCE COLUMN VALIDATION
# ============================================================

source_column_names = {
    str(c).upper(): c
    for c in df.columns
}


def validate_source_column(
    column_name,
    parameter_name
):

    if column_name is None:

        raise ValueError(
            f"Parameter '{parameter_name}' "
            f"resolved to NULL."
        )

    column_string = (
        str(column_name).strip()
    )

    if (
        column_string.upper()
        not in source_column_names
    ):

        raise ValueError(
            f"Parameter '{parameter_name}' "
            f"references source column "
            f"'{column_string}', but that column "
            f"does not exist in asset "
            f"'{asset_name}'. "
            f"Available columns: "
            f"{list(df.columns)}"
        )

    return source_column_names[
        column_string.upper()
    ]


# ============================================================
# 10. REQUIRED COLUMNS
# ============================================================

if "required_columns" in parsed_parameters:

    required_columns = (
        parsed_parameters[
            "required_columns"
        ]
    )

    if (
        not isinstance(
            required_columns,
            list
        )
        or not required_columns
    ):

        raise ValueError(
            "required_columns must be "
            "a non-empty JSON array."
        )

    required_conditions = []

    for col in required_columns:

        actual_col = (
            validate_source_column(
                col,
                "required_columns"
            )
        )

        quoted = sql_identifier(
            actual_col
        )

        required_conditions.append(
            f"{quoted} IS NOT NULL "
            f"AND TRIM("
            f"CAST({quoted} AS STRING)"
            f") <> ''"
        )

    execution_context[
        "required_columns_condition"
    ] = " AND ".join(
        required_conditions
    )


# ============================================================
# 11. SELECTED COLUMNS
# ============================================================

if "selected_columns" in parsed_parameters:

    selected_columns = (
        parsed_parameters[
            "selected_columns"
        ]
    )

    if (
        not isinstance(
            selected_columns,
            list
        )
        or not selected_columns
    ):

        raise ValueError(
            "selected_columns must be "
            "a non-empty JSON array."
        )

    selected_conditions = []

    for col in selected_columns:

        actual_col = (
            validate_source_column(
                col,
                "selected_columns"
            )
        )

        quoted = sql_identifier(
            actual_col
        )

        selected_conditions.append(
            f"{quoted} IS NOT NULL "
            f"AND TRIM("
            f"CAST({quoted} AS STRING)"
            f") <> ''"
        )

    execution_context[
        "selected_columns_condition"
    ] = " AND ".join(
        selected_conditions
    )


# ============================================================
# 12. DATE SEQUENCE COLUMNS
# ============================================================

if "date_sequence_columns" in parsed_parameters:

    date_sequence_columns = (
        parsed_parameters[
            "date_sequence_columns"
        ]
    )

    if (
        not isinstance(
            date_sequence_columns,
            list
        )
        or len(date_sequence_columns) < 2
    ):

        raise ValueError(
            "date_sequence_columns must "
            "contain at least two source "
            "columns."
        )

    normalized_date_columns = [

        validate_source_column(
            col,
            "date_sequence_columns"
        )

        for col in date_sequence_columns
    ]

    date_conditions = []

    for i in range(
        len(normalized_date_columns) - 1
    ):

        current_column = sql_identifier(
            normalized_date_columns[i]
        )

        next_column = sql_identifier(
            normalized_date_columns[i + 1]
        )

        date_conditions.append(
            f"TRY_CAST("
            f"{current_column} AS TIMESTAMP"
            f") <= TRY_CAST("
            f"{next_column} AS TIMESTAMP"
            f")"
        )

    execution_context[
        "date_sequence_condition"
    ] = " AND ".join(
        date_conditions
    )


# ============================================================
# 13. IDENTIFY DATE/TIME PARAMETER COLUMNS
# ============================================================

date_time_parameter_names = []


if (
    data_type is not None
    and str(data_type)
        .strip()
        .upper()
    in {
        "DATE",
        "TIMESTAMP",
        "DATE/TIMESTAMP"
    }
):

    if parameter_definition_obj:

        if isinstance(
            parameter_definition_obj,
            dict
        ):

            parameter_definitions = (
                parameter_definition_obj.get(
                    "parameters",
                    []
                )
            )

        elif isinstance(
            parameter_definition_obj,
            list
        ):

            parameter_definitions = (
                parameter_definition_obj
            )

        else:

            parameter_definitions = []

        for parameter in (
            parameter_definitions
        ):

            if not isinstance(
                parameter,
                dict
            ):
                continue

            parameter_name = (
                parameter.get("name")
            )

            control = parameter.get(
                "control",
                "text"
            )

            if (
                parameter_name
                and control in {
                    "column_select",
                    "column"
                }
                and parameter_name
                in parsed_parameters
            ):

                date_time_parameter_names.append(
                    parameter_name
                )


execution_context[
    "date_time_parameter_names"
] = (
    date_time_parameter_names
)


# ============================================================
# 14. SQL TEMPLATE PLACEHOLDER METADATA
#
# IMPORTANT:
# Regex quantifiers such as:
#
#     ^REV[0-9]{8}$
#
# are NOT placeholders.
#
# This set is used by the SQL rendering cell.
# ============================================================

supported_sql_placeholders = {

    "table",
    "asset",
    "column",

    "source_relation",

    "target_column_id",
    "target_column_name",
    "target_column_sql",

    "asset_id",
    "asset_name",
    "asset_type",

    "assignment_id",
    "rule_code",
    "version_number",

    "dimension",
    "rule_name",
    "rule_type",
    "target_type",

    "parameter_values",
    "sql_expression",

    "threshold_value",
    "evaluation_operation",

    "data_type",
    "semantic_type",

    "required_columns_condition",
    "selected_columns_condition",
    "date_sequence_condition"
}


# ============================================================
# 15. DISPLAY EXECUTION VALUES
# ============================================================

print("")
print("EXECUTION VALUES")
print("------------------------------------")

print(
    f"Source relation          : "
    f"{source_relation}"
)

print(
    f"Target column            : "
    f"{target_column_name}"
)

print(
    f"Effective threshold      : "
    f"{effective_threshold}"
)

print(
    f"Evaluation operation     : "
    f"{effective_evaluation_operation}"
)

print(
    f"Date/time parameters     : "
    f"{date_time_parameter_names}"
)

print("")
print("Execution context created")
print("------------------------------------")


for key, value in execution_context.items():

    if key == "parameter_values":

        print(
            f"{key:25} : "
            f"{json.dumps(value, default=str)}"
        )

    else:

        print(
            f"{key:25} : {value}"
        )

BUILDING EXECUTION CONTEXT

EXECUTION VALUES
------------------------------------
Source relation          : dq_source_data
Target column            : STATUS_ID
Effective threshold      : 99.0
Evaluation operation     : >=
Date/time parameters     : []

Execution context created
------------------------------------
table                     : dq_source_data
asset                     : dq_source_data
column                    : `STATUS_ID`
asset_id                  : 1
asset_name                : 03_ISS_account_status
asset_type                : TABLE
assignment_id             : 926
rule_code                 : V08
version_number            : 1
dimension                 : VALIDITY
rule_name                 : Format compliance %
rule_type                 : PARAMETERIZED
target_type               : COLUMN
target_column_id          : 504
target_column_name        : STATUS_ID
target_column_sql         : `STATUS_ID`
source_relation           : dq_source_data
parameter_values          : {"form

In [0]:
# ============================================================
# RENDER SQL TEMPLATE
# ============================================================

print("====================================")
print("RENDERING SQL TEMPLATE")
print("====================================")


# ============================================================
# 1. VALIDATE SQL TEMPLATE
# ============================================================

if not sql_query_template:

    raise ValueError(
        f"No sql_query_template found for rule {rule_code}."
    )


# ============================================================
# 2. SQL HELPER FUNCTIONS
# ============================================================

def sql_string(value):

    if value is None:
        return "NULL"

    return (
        "'"
        + str(value).replace("'", "''")
        + "'"
    )


def sql_identifier(value):

    if value is None:
        raise ValueError(
            "SQL identifier cannot be NULL."
        )

    value = str(value).strip()

    if not value:

        raise ValueError(
            "SQL identifier cannot be empty."
        )

    return (
        "`"
        + value.replace("`", "``")
        + "`"
    )


def sql_number(value):

    if value is None:

        raise ValueError(
            "Numeric parameter cannot be NULL."
        )

    value_string = str(value).strip()

    if not re.fullmatch(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)",
        value_string
    ):

        raise ValueError(
            f"Invalid numeric SQL value: "
            f"{value_string}"
        )

    return value_string


def sql_multi_value(value):

    if (
        not isinstance(value, list)
        or not value
    ):

        raise ValueError(
            "Multi-value parameter must be "
            "a non-empty JSON array."
        )

    return ", ".join(
        sql_string(v)
        for v in value
    )


def sql_timestamp_identifier(value):

    actual_column = validate_source_column(
        value,
        "timestamp_column"
    )

    return (
        f"TRY_CAST("
        f"{sql_identifier(actual_column)} "
        f"AS TIMESTAMP)"
    )


# ============================================================
# 3. START WITH METADATA SQL TEMPLATE
# ============================================================

rendered_sql = sql_query_template


# ============================================================
# 4. STANDARD PLACEHOLDERS
# ============================================================

standard_placeholders = {

    "{table}":
        source_relation,

    "{asset}":
        source_relation,

    "{column}":
        target_column_sql
}


for token, value in standard_placeholders.items():

    if token not in rendered_sql:
        continue

    if value is None:

        raise ValueError(
            f"Placeholder {token} requires a value."
        )

    rendered_sql = rendered_sql.replace(
        token,
        str(value)
    )


# ============================================================
# 5. PARAMETERIZED PLACEHOLDERS
# ============================================================

if parameter_definition_obj:

    if isinstance(
        parameter_definition_obj,
        dict
    ):

        parameter_definitions = (
            parameter_definition_obj.get(
                "parameters",
                []
            )
        )

    elif isinstance(
        parameter_definition_obj,
        list
    ):

        parameter_definitions = (
            parameter_definition_obj
        )

    else:

        parameter_definitions = []


    for parameter in parameter_definitions:

        if not isinstance(
            parameter,
            dict
        ):

            continue


        parameter_name = (
            parameter.get("name")
        )

        control = parameter.get(
            "control",
            "text"
        )


        if not parameter_name:
            continue


        token = (
            "{"
            + parameter_name
            + "}"
        )


        # Parameter is not used by this
        # particular SQL template.
        if token not in rendered_sql:
            continue


        value = parsed_parameters.get(
            parameter_name
        )


        if value is None:

            raise ValueError(
                f"Parameter '{parameter_name}' "
                f"is required by the SQL template "
                f"but no value was supplied."
            )


        # ----------------------------------------------------
        # COLUMN PARAMETER
        # ----------------------------------------------------

        if control in {
            "column_select",
            "column"
        }:

            if (
                parameter_name
                in date_time_parameter_names
            ):

                rendered_value = (
                    sql_timestamp_identifier(
                        value
                    )
                )

            else:

                actual_col = (
                    validate_source_column(
                        value,
                        parameter_name
                    )
                )

                rendered_value = (
                    sql_identifier(
                        actual_col
                    )
                )


        # ----------------------------------------------------
        # MULTI-COLUMN PARAMETER
        # ----------------------------------------------------

        elif control in {
            "column_multi_select",
            "ordered_column_select"
        }:

            if (
                not isinstance(
                    value,
                    list
                )
                or not value
            ):

                raise ValueError(
                    f"Parameter "
                    f"'{parameter_name}' must be "
                    f"a non-empty JSON array."
                )


            if (
                parameter_name
                in date_time_parameter_names
            ):

                rendered_value = ", ".join(

                    sql_timestamp_identifier(
                        v
                    )

                    for v in value
                )

            else:

                rendered_value = ", ".join(

                    sql_identifier(
                        validate_source_column(
                            v,
                            parameter_name
                        )
                    )

                    for v in value
                )


        # ----------------------------------------------------
        # NUMBER
        # ----------------------------------------------------

        elif control == "number":

            rendered_value = (
                sql_number(value)
            )


        # ----------------------------------------------------
        # TEXT / REGEX / DATETIME
        #
        # IMPORTANT:
        # Regex values such as:
        #
        #     ^REV[0-9]{8}$
        #
        # are rendered as SQL string literals.
        #
        # The {8} is NOT an SQL placeholder.
        # ----------------------------------------------------

        elif control in {
            "text",
            "regex",
            "datetime"
        }:

            rendered_value = (
                sql_string(value)
            )


        # ----------------------------------------------------
        # MULTI-VALUE
        # ----------------------------------------------------

        elif control == "multi_value":

            rendered_value = (
                sql_multi_value(value)
            )


        # ----------------------------------------------------
        # EXPRESSION
        # ----------------------------------------------------

        elif control in {
            "expression",
            "expression_multi"
        }:

            rendered_value = str(value)


        # ----------------------------------------------------
        # SELECT / ASSET SELECT
        # ----------------------------------------------------

        elif control in {
            "select",
            "asset_select"
        }:

            rendered_value = str(value)


        # ----------------------------------------------------
        # JSON / SCHEMA
        # ----------------------------------------------------

        elif control in {
            "json",
            "schema_definition"
        }:

            rendered_value = sql_string(
                json.dumps(
                    value,
                    default=str
                )
            )


        # ----------------------------------------------------
        # DEFAULT
        # ----------------------------------------------------

        else:

            rendered_value = (
                sql_string(value)
            )


        rendered_sql = rendered_sql.replace(
            token,
            rendered_value
        )


# ============================================================
# 6. DERIVED EXECUTION-CONTEXT PLACEHOLDERS
# ============================================================

for placeholder, value in (
    execution_context.items()
):

    token = (
        "{"
        + placeholder
        + "}"
    )


    if token not in rendered_sql:
        continue


    if value is None:

        raise ValueError(
            f"Placeholder {token} "
            f"requires a value."
        )


    # Do not replace a parameter twice.
    if placeholder in parsed_parameters:
        continue


    # This is execution metadata, not SQL.
    if (
        placeholder
        == "date_time_parameter_names"
    ):

        continue


    rendered_sql = rendered_sql.replace(
        token,
        str(value)
    )


# ============================================================
# 7. CURRENT PROTOTYPE SAFETY OVERRIDES
#
# The Excel source is intentionally loaded as STRING.
#
# Therefore numeric/date rules need TRY_CAST so malformed
# source values do not terminate the entire DQ execution.
# ============================================================

safe_numeric_column = (

    f"TRY_CAST("
    f"{target_column_sql} "
    f"AS DECIMAL(38,10))"

    if target_column_sql is not None
    else None
)


safe_timestamp_column = (

    f"TRY_CAST("
    f"{target_column_sql} "
    f"AS TIMESTAMP)"

    if target_column_sql is not None
    else None
)


# ============================================================
# P04 / P05 / P06
# ============================================================

if rule_code in {
    "P04",
    "P05",
    "P06"
}:

    if safe_numeric_column is None:

        raise ValueError(
            f"{rule_code} requires "
            f"a target column."
        )


    if rule_code == "P04":

        rendered_sql = f"""
SELECT
    CAST(
        MIN({safe_numeric_column})
        AS DOUBLE
    ) AS metric_value,

    0 AS passed_count,
    0 AS failed_count,

    COUNT(*) AS total_count,

    COUNT({safe_numeric_column})
        AS applicable_count

FROM {source_relation}
""".strip()


    elif rule_code == "P05":

        rendered_sql = f"""
SELECT
    CAST(
        MAX({safe_numeric_column})
        AS DOUBLE
    ) AS metric_value,

    0 AS passed_count,
    0 AS failed_count,

    COUNT(*) AS total_count,

    COUNT({safe_numeric_column})
        AS applicable_count

FROM {source_relation}
""".strip()


    else:

        rendered_sql = f"""
SELECT
    CAST(
        AVG({safe_numeric_column})
        AS DOUBLE
    ) AS metric_value,

    0 AS passed_count,
    0 AS failed_count,

    COUNT(*) AS total_count,

    COUNT({safe_numeric_column})
        AS applicable_count

FROM {source_relation}
""".strip()


# ============================================================
# V10 / V11
# ============================================================

elif rule_code in {
    "V10",
    "V11"
}:

    if safe_numeric_column is None:

        raise ValueError(
            f"{rule_code} requires "
            f"a target column."
        )


    if rule_code == "V10":

        comparison = (
            f"{safe_numeric_column} > 0"
        )

    else:

        comparison = (
            f"{safe_numeric_column} >= 0"
        )


    rendered_sql = f"""
SELECT

    100.0 * COUNT_IF(
        {comparison}
    )
    / NULLIF(
        COUNT({target_column_sql}),
        0
    ) AS metric_value,

    COUNT_IF(
        {comparison}
    ) AS passed_count,

    COUNT({target_column_sql})
        - COUNT_IF({comparison})
        AS failed_count,

    COUNT({target_column_sql})
        AS total_count,

    COUNT({target_column_sql})
        AS applicable_count

FROM {source_relation}
""".strip()


# ============================================================
# V12
# ============================================================

elif rule_code == "V12":

    if safe_timestamp_column is None:

        raise ValueError(
            "V12 requires a target column."
        )


    rendered_sql = f"""
SELECT

    100.0 * COUNT_IF(
        {safe_timestamp_column}
        > CURRENT_TIMESTAMP()
    )
    / NULLIF(
        COUNT({target_column_sql}),
        0
    ) AS metric_value,

    COUNT_IF(
        {safe_timestamp_column}
        <= CURRENT_TIMESTAMP()
    ) AS passed_count,

    COUNT_IF(
        {safe_timestamp_column}
        > CURRENT_TIMESTAMP()
    ) AS failed_count,

    COUNT({target_column_sql})
        AS total_count,

    COUNT({target_column_sql})
        AS applicable_count

FROM {source_relation}
""".strip()


# ============================================================
# 8. SAFE UNRESOLVED-PLACEHOLDER VALIDATION
#
# IMPORTANT:
#
# Do NOT use:
#
#     r"\{([A-Za-z0-9_]+)\}"
#
# because this incorrectly treats regex quantifiers such as
#
#     {8}
#     {10}
#     {2}
#
# as SQL placeholders.
#
# Instead, only known executor placeholders are considered.
# ============================================================

supported_sql_placeholders = {

    "table",
    "asset",
    "column",

    "source_relation",

    "asset_id",
    "asset_name",
    "asset_type",

    "assignment_id",
    "rule_code",
    "version_number",

    "dimension",
    "rule_name",
    "rule_type",
    "target_type",

    "target_column_id",
    "target_column_name",
    "target_column_sql",

    "parameter_values",
    "sql_expression",

    "threshold_value",
    "evaluation_operation",

    "data_type",
    "semantic_type",

    "required_columns_condition",
    "selected_columns_condition",

    "date_sequence_condition"
}


unresolved_placeholders = []


for match in re.finditer(
    r"\{([^{}]+)\}",
    rendered_sql
):

    placeholder_name = (
        match.group(1).strip()
    )


    # Only placeholders that belong to the
    # executor are considered unresolved.
    #
    # Examples ignored correctly:
    #
    # {8}
    # {10}
    # {2}
    #
    # because these are regex quantifiers.

    if (
        placeholder_name
        in supported_sql_placeholders
    ):

        unresolved_placeholders.append(
            placeholder_name
        )


if unresolved_placeholders:

    raise ValueError(
        f"Unresolved SQL placeholders "
        f"for rule {rule_code}: "
        f"{sorted(set(unresolved_placeholders))}"
    )


# ============================================================
# 9. DISPLAY FINAL SQL
# ============================================================

print("")
print("RENDERED SQL")
print("------------------------------------")

print(rendered_sql)

print("")
print("SQL rendering completed")
print("------------------------------------")

RENDERING SQL TEMPLATE

RENDERED SQL
------------------------------------

SELECT
    100.0 * COUNT_IF(
        REGEXP_LIKE(`STATUS_ID`, '^AST[0-9]{8}$')
    )
    / NULLIF(COUNT(`STATUS_ID`), 0) AS metric_value,

    COUNT_IF(
        REGEXP_LIKE(`STATUS_ID`, '^AST[0-9]{8}$')
    ) AS passed_count,

    COUNT_IF(
        NOT REGEXP_LIKE(`STATUS_ID`, '^AST[0-9]{8}$')
    ) AS failed_count,

    COUNT(`STATUS_ID`) AS total_count,

    COUNT(`STATUS_ID`) AS applicable_count

FROM dq_source_data

WHERE `STATUS_ID` IS NOT NULL


SQL rendering completed
------------------------------------


In [0]:
# ============================================================
# EXECUTE RULE SQL
# ============================================================

print("====================================")
print("EXECUTING RULE SQL")
print("====================================")

try:

    rule_result_df = spark.sql(
        rendered_sql
    )

except Exception as e:

    raise RuntimeError(
        f"Rule SQL execution failed for "
        f"rule_code='{rule_code}', "
        f"assignment_id={assignment_id_int}.\n"
        f"Rendered SQL:\n{rendered_sql}\n"
        f"Error: {str(e)}"
    )

result_columns = rule_result_df.columns

required_result_columns = [
    "metric_value",
    "passed_count",
    "failed_count",
    "total_count",
    "applicable_count"
]

missing_result_columns = [
    column
    for column in required_result_columns
    if column not in result_columns
]

if missing_result_columns:
    raise ValueError(
        "Rule SQL did not return the required standard result "
        f"columns: {missing_result_columns}. "
        f"Returned columns: {result_columns}"
    )

result_row_count = rule_result_df.count()

if result_row_count != 1:
    raise ValueError(
        "Rule SQL must return exactly one result row. "
        f"Returned {result_row_count} rows."
    )

rule_result = rule_result_df.first()

metric_value = (
    float(rule_result["metric_value"])
    if rule_result["metric_value"] is not None
    else None
)

passed_count = (
    int(rule_result["passed_count"])
    if rule_result["passed_count"] is not None
    else 0
)

failed_count = (
    int(rule_result["failed_count"])
    if rule_result["failed_count"] is not None
    else 0
)

total_count = (
    int(rule_result["total_count"])
    if rule_result["total_count"] is not None
    else 0
)

applicable_count = (
    int(rule_result["applicable_count"])
    if rule_result["applicable_count"] is not None
    else 0
)

# ------------------------------------------------------------
# Identify malformed typed values.
#
# NULL is missing, not malformed.
# Non-NULL values that cannot safely cast to the rule's expected
# type are malformed.
# ------------------------------------------------------------

malformed_count = 0
date_time_total_count = 0
date_time_applicable_count = 0
numeric_malformed_count = 0

# DATE/TIMESTAMP parameter validation.
if date_time_parameter_names:

    malformed_conditions = []
    non_null_conditions = []
    valid_conditions = []

    for parameter_name in date_time_parameter_names:

        source_column = validate_source_column(
            parsed_parameters.get(parameter_name),
            parameter_name
        )

        quoted_column = sql_identifier(source_column)

        malformed_conditions.append(
            "("
            f"{quoted_column} IS NOT NULL "
            f"AND TRY_CAST({quoted_column} AS TIMESTAMP) IS NULL"
            ")"
        )

        non_null_conditions.append(
            f"{quoted_column} IS NOT NULL"
        )

        valid_conditions.append(
            f"TRY_CAST({quoted_column} AS TIMESTAMP) IS NOT NULL"
        )

    if malformed_conditions:

        malformed_count = int(
            spark.sql(
                f"""
                SELECT COUNT(*) AS malformed_count
                FROM {SOURCE_VIEW}
                WHERE {" OR ".join(malformed_conditions)}
                """
            ).first()["malformed_count"]
        )

    if non_null_conditions:

        date_time_applicable_count = int(
            spark.sql(
                f"""
                SELECT COUNT(*) AS applicable_count
                FROM {SOURCE_VIEW}
                WHERE {" AND ".join(non_null_conditions)}
                """
            ).first()["applicable_count"]
        )

    if valid_conditions:

        date_time_total_count = int(
            spark.sql(
                f"""
                SELECT COUNT(*) AS valid_count
                FROM {SOURCE_VIEW}
                WHERE {" AND ".join(valid_conditions)}
                """
            ).first()["valid_count"]
        )

# Numeric validity rules: invalid non-null numeric values are failures.
if rule_code in {"V10", "V11"} and target_column_name is not None:

    quoted_target = sql_identifier(target_column_name)

    numeric_malformed_count = int(
        spark.sql(
            f"""
            SELECT COUNT(*) AS malformed_count
            FROM {SOURCE_VIEW}
            WHERE {quoted_target} IS NOT NULL
              AND TRY_CAST(
                    {quoted_target} AS DECIMAL(38,10)
                  ) IS NULL
            """
        ).first()["malformed_count"]
    )

    # The V10/V11 SQL already includes invalid numeric values in
    # failed_count through COUNT(raw) - COUNT_IF(valid condition).
    # Do not add them a second time.
    malformed_count = numeric_malformed_count

# For numeric profiling, malformed values are excluded from the
# metric's applicable population and do not automatically make
# a metric-only profiling rule FAIL.
if rule_code in {"P04", "P05", "P06"}:
    malformed_count = 0

# ------------------------------------------------------------
# Calculate failure percentage.
# ------------------------------------------------------------

if total_count > 0:
    failure_percentage = (
        100.0
        * failed_count
        / total_count
    )
else:
    failure_percentage = 0.0

print("Rule SQL executed successfully")
print("------------------------------------")
print(f"metric_value             : {metric_value}")
print(f"passed_count             : {passed_count}")
print(f"failed_count             : {failed_count}")
print(f"total_count              : {total_count}")
print(f"applicable_count         : {applicable_count}")
print(f"failure_percentage       : {failure_percentage:.2f}")

print("")
print("DATA TYPE QUALITY")
print("------------------------------------")
print(
    f"date/time parameters     : "
    f"{date_time_parameter_names}"
)
print(
    f"valid timestamp rows     : "
    f"{date_time_total_count}"
)
print(
    f"non-null timestamp rows  : "
    f"{date_time_applicable_count}"
)
print(
    f"malformed timestamp rows : "
    f"{malformed_count if date_time_parameter_names else 0}"
)
print(
    f"malformed numeric rows   : "
    f"{numeric_malformed_count}"
)


EXECUTING RULE SQL
Rule SQL executed successfully
------------------------------------
metric_value             : 100.0
passed_count             : 100
failed_count             : 0
total_count              : 100
applicable_count         : 100
failure_percentage       : 0.00

DATA TYPE QUALITY
------------------------------------
date/time parameters     : []
valid timestamp rows     : 0
non-null timestamp rows  : 0
malformed timestamp rows : 0
malformed numeric rows   : 0


In [0]:
# ============================================================
# EVALUATE RULE STATUS
# ============================================================

print("====================================")
print("EVALUATING RULE RESULT")
print("====================================")

metric_evaluation_passed = None
threshold_passed = None

# --------------------------------------------------
# Metric-only / profiling rule.
#
# threshold=NULL is valid and means:
# execute the rule and return the metric, but do not
# manufacture a PASS/FAIL result.
# --------------------------------------------------

if effective_threshold is None:

    evaluation_status = "PASS"

    metric_evaluation_passed = None
    threshold_passed = None

    print("No threshold configured.")
    print(
        "Rule executed successfully. "
        "PASS/FAIL threshold evaluation was not performed."
    )

else:

    if metric_value is None:

        evaluation_status = "ERROR"
        metric_evaluation_passed = False
        threshold_passed = False

        print(
            "Metric value is NULL. "
            "The threshold cannot be evaluated."
        )

    else:

        if effective_evaluation_operation == ">":
            threshold_passed = (
                metric_value > effective_threshold
            )

        elif effective_evaluation_operation == ">=":
            threshold_passed = (
                metric_value >= effective_threshold
            )

        elif effective_evaluation_operation == "<":
            threshold_passed = (
                metric_value < effective_threshold
            )

        elif effective_evaluation_operation == "<=":
            threshold_passed = (
                metric_value <= effective_threshold
            )

        elif effective_evaluation_operation in {"=", "=="}:
            threshold_passed = (
                metric_value == effective_threshold
            )

        elif effective_evaluation_operation == "!=":
            threshold_passed = (
                metric_value != effective_threshold
            )

        else:
            raise ValueError(
                f"Unsupported evaluation operation: "
                f"'{effective_evaluation_operation}'"
            )

        metric_evaluation_passed = threshold_passed

        evaluation_status = (
            "PASS"
            if threshold_passed
            else "FAIL"
        )

# --------------------------------------------------
# Malformed typed data can force a threshold-based
# rule to FAIL.
#
# Metric-only rules remain NO_THRESHOLD unless SQL
# execution itself failed.
# --------------------------------------------------

if effective_threshold is not None and malformed_count > 0:
    evaluation_status = "FAIL"

print("Rule evaluation completed")
print("------------------------------------")
print(
    f"Metric value              : "
    f"{metric_value}"
)
print(
    f"Threshold                 : "
    f"{effective_threshold}"
)
print(
    f"Evaluation operation      : "
    f"{effective_evaluation_operation}"
)

if effective_threshold is None:
    print(
        "Metric evaluation         : "
        "NOT PERFORMED"
    )
else:
    print(
        f"Metric evaluation         : "
        f"{'PASS' if metric_evaluation_passed else 'FAIL'}"
    )

print(
    f"Malformed data rows       : "
    f"{malformed_count}"
)
print(
    f"Final rule status         : "
    f"{evaluation_status}"
)


EVALUATING RULE RESULT
Rule evaluation completed
------------------------------------
Metric value              : 100.0
Threshold                 : 99.0
Evaluation operation      : >=
Metric evaluation         : PASS
Malformed data rows       : 0
Final rule status         : PASS


In [0]:
print("====================================")
print("WRITING RULE EXECUTION RESULT")
print("====================================")

result_details = {
    "rule_code": rule_code,
    "rule_name": rule_name,
    "version_number": version_number_int,
    "target_column_id": target_column_id_int,
    "target_column_name": target_column_name,

    "metric_value": metric_value,
    "threshold_value": effective_threshold,
    "evaluation_operation": effective_evaluation_operation,
    "evaluation_status": evaluation_status,

    "passed_count": passed_count,
    "failed_count": failed_count,
    "total_count": total_count,
    "applicable_count": applicable_count,

    "malformed_count": malformed_count,
    "numeric_malformed_count": numeric_malformed_count,
    "date_time_parameter_names": date_time_parameter_names,

    "parameter_values": parsed_parameters
}

result_details_json = json.dumps(
    result_details,
    default=str
)

asset_execution_id = globals().get(
    "asset_execution_id",
    None
)

if asset_execution_id in (None, "", "None"):

    print("No asset_execution_id supplied.")
    print(
        "Standalone test mode: result will not be inserted "
        "into dq_execution_results_m."
    )

    result_written = False

else:

    try:
        asset_execution_id_int = int(
            asset_execution_id
        )
    except ValueError:
        raise ValueError(
            "asset_execution_id must be a valid integer."
        )

    failure_percentage_value = (
        Decimal(str(round(failure_percentage, 2)))
        if failure_percentage is not None
        else None
    )

    threshold_value_value = (
        Decimal(str(effective_threshold))
        if effective_threshold is not None
        else None
    )

    result_df = spark.createDataFrame(
        [
            (
                asset_execution_id_int,
                rule_code,
                None,
                assignment_id_int,
                dimension,
                evaluation_status,
                int(total_count)
                if total_count is not None
                else None,
                int(passed_count)
                if passed_count is not None
                else None,
                int(failed_count)
                if failed_count is not None
                else None,
                failure_percentage_value,
                threshold_value_value,
                result_details_json
            )
        ],
        T.StructType([
            T.StructField(
                "asset_execution_id",
                T.LongType(),
                False
            ),
            T.StructField(
                "rule_code",
                T.StringType(),
                True
            ),
            T.StructField(
                "rule_version_id",
                T.LongType(),
                True
            ),
            T.StructField(
                "assignment_id",
                T.LongType(),
                True
            ),
            T.StructField(
                "dimension",
                T.StringType(),
                False
            ),
            T.StructField(
                "status",
                T.StringType(),
                False
            ),
            T.StructField(
                "records_evaluated",
                T.LongType(),
                True
            ),
            T.StructField(
                "records_passed",
                T.LongType(),
                True
            ),
            T.StructField(
                "records_failed",
                T.LongType(),
                True
            ),
            T.StructField(
                "failure_percentage",
                T.DecimalType(7, 2),
                True
            ),
            T.StructField(
                "threshold_value",
                T.DecimalType(18, 6),
                True
            ),
            T.StructField(
                "result_details",
                T.StringType(),
                True
            )
        ])
    )

    result_df.createOrReplaceTempView(
        "dq_rule_result_to_write"
    )

    # Keep rule_version_id NULL because version information is now
    # represented by rules_assignments_m.version_number.
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.dq_execution_results_m
        (
            asset_execution_id,
            rule_code,
            rule_version_id,
            assignment_id,
            dimension,
            status,
            records_evaluated,
            records_passed,
            records_failed,
            failure_percentage,
            threshold_value,
            result_details
        )
        SELECT
            asset_execution_id,
            rule_code,
            rule_version_id,
            assignment_id,
            dimension,
            status,
            records_evaluated,
            records_passed,
            records_failed,
            failure_percentage,
            threshold_value,
            result_details
        FROM dq_rule_result_to_write
    """)

    result_written = True

    print(
        "Result successfully written to "
        "dq_execution_results_m."
    )

print("")
print("RESULT SUMMARY")
print("------------------------------------")
print(f"Rule code          : {rule_code}")
print(f"Assignment ID      : {assignment_id_int}")
print(f"Status             : {evaluation_status}")
print(f"Metric value       : {metric_value}")
print(f"Threshold          : {effective_threshold}")
print(f"Records evaluated  : {total_count}")
print(f"Records passed     : {passed_count}")
print(f"Records failed     : {failed_count}")
print(f"Result written     : {result_written}")


WRITING RULE EXECUTION RESULT
Result successfully written to dq_execution_results_m.

RESULT SUMMARY
------------------------------------
Rule code          : V08
Assignment ID      : 926
Status             : PASS
Metric value       : 100.0
Threshold          : 99.0
Records evaluated  : 100
Records passed     : 100
Records failed     : 0
Result written     : True


In [0]:
# ============================================================
# RETURN RESULT TO ORCHESTRATOR
# ============================================================

result = {
    "status": evaluation_status,
    "rule_code": rule_code,
    "assignment_id": assignment_id_int,
    "version_number": version_number_int,
    "metric_value": metric_value,
    "threshold_value": effective_threshold,
    "evaluation_operation": effective_evaluation_operation,
    "records_evaluated": total_count,
    "records_passed": passed_count,
    "records_failed": failed_count,
    "applicable_count": applicable_count,
    "failure_percentage": failure_percentage,
    "malformed_count": malformed_count,
    "result_written": result_written
}

print("")
print("====================================")
print("RULE EXECUTION COMPLETE")
print("====================================")
print(json.dumps(result, default=str, indent=2))

dbutils.notebook.exit(
    json.dumps(result, default=str)
)
